# 02 — Data split, baselines and char-CNN training
Input: `data/labeled.csv` (output of notebook 01).
Output: `data/train.csv`, `data/val.csv`, `data/test.csv` and the trained model in `model/`.

The test set is **not** used in this notebook.

In [ ]:
import os, sys
# Colab: clone the repository first
if "google.colab" in sys.modules and not os.path.exists("src"):
    !git clone https://github.com/<username>/kz-profanity-detection-DL26.git
    %cd kz-profanity-detection-DL26
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Working dir:", os.getcwd())

In [ ]:
import json, random, time
import numpy as np, pandas as pd
import torch, torch.nn as nn
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support

from src.preprocess import normalize, group_key
from src.model import CharVocab, CharCNN

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
set_seed()


## 1. Labeled data

In [ ]:
df = pd.read_csv("data/labeled.csv")
print("Rows:", len(df))
print("Label method:", df.label_method.value_counts().to_dict())
print("Uncertain rows (dropped):", int(df.uncertain.sum()))
df = df[df.uncertain != 1].copy()
df["norm"] = df.text.map(normalize)

before = len(df)
df = df.drop_duplicates("norm")                      # exact duplicates after normalization
print("Removed duplicates:", before - len(df))
df["group_id"] = df.text.map(group_key)             # near-duplicates share a group
print("Rows:", len(df), "| groups:", df.group_id.nunique())
print(df.label.value_counts().rename({0: "clean (0)", 1: "profane (1)"}))

## 2. Split 70 / 15 / 15
`StratifiedGroupKFold` with 20 folds: 14 folds → train, 3 → validation, 3 → test.
- **Stratified**: each split keeps a similar share of profane comments.
- **Grouped**: near-duplicates (same letters, ignoring digits/punctuation/emoji) never end up in different splits.

Only manually labeled rows (`label_method == "manual"`) are allowed in the test set; other test rows are moved to train.

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=SEED)
fold = np.zeros(len(df), dtype=int)
for k, (_, idx) in enumerate(sgkf.split(df, df.label, groups=df.group_id)):
    fold[idx] = k
df["split"] = np.select([fold < 14, fold < 17], ["train", "val"], "test")

if (df.label_method == "manual").any():               # test set: manually labeled rows only
    move = (df.split == "test") & (df.label_method != "manual")
    df.loc[move, "split"] = "train"
    print("Moved", int(move.sum()), "non-manual rows from test to train")

assert not set(df[df.split == "train"].group_id) & set(df[df.split == "test"].group_id)
assert not set(df[df.split == "train"].group_id) & set(df[df.split == "val"].group_id)

summary = df.groupby("split").label.agg(total="size", profane="sum")
summary["clean"] = summary.total - summary.profane
summary["% profane"] = (100 * summary.profane / summary.total).round(1)
print(summary.loc[["train", "val", "test"]])

cols = ["custom_id", "text", "label", "domain", "group", "matched", "synthetic", "label_method", "group_id"]
for s in ["train", "val", "test"]:
    df[df.split == s][cols].to_csv(f"data/{s}.csv", index=False)
train, val = df[df.split == "train"], df[df.split == "val"]

## 3. Comment length and `MAX_LEN`

In [ ]:
lens = train.norm.str.len()
print(lens.describe(percentiles=[.5, .9, .95, .99]).round(0))
MAX_LEN = 300
print(f"Comments longer than {MAX_LEN} chars: {(lens > MAX_LEN).mean():.1%} (they are truncated)")

## 4. Baselines (validation set only)
1. **Keyword rule**: predict 1 if the word list from notebook 01 matched.
2. **TF-IDF character n-grams + logistic regression**, fitted on train only.

Note: most labels originate from the keyword search, so the keyword rule is expected to score high here.
This is a limitation of the data construction (see README).

In [ ]:
def prf(y, p):
    P, R, F, _ = precision_recall_fscore_support(y, p, average="binary", zero_division=0)
    return {"precision": round(P, 3), "recall": round(R, 3), "f1": round(F, 3)}

results = {}
results["keyword rule"] = prf(val.label, (val.group == "candidate").astype(int))

tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2)
Xtr, Xva = tfidf.fit_transform(train.norm), tfidf.transform(val.norm)
lr = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED).fit(Xtr, train.label)
results["TF-IDF + LogReg"] = prf(val.label, lr.predict(Xva))
pd.DataFrame(results).T

## 5. Char-CNN
**How text becomes input:** normalized text → each character → an integer ID from the training vocabulary.
Characters seen fewer than 2 times in train become `<unk>`. Comments are cut or padded to `MAX_LEN`.
Punctuation and emoji are kept as characters, because masks like `с*ка` use them.

**Objective:** binary cross-entropy on the logit, with `pos_weight` = #clean / #profane to balance the classes.

In [ ]:
CFG = {
    "seed": SEED, "max_len": MAX_LEN, "min_char_freq": 2,
    "model": {"emb_dim": 48, "n_filters": 96, "kernel_sizes": [3, 4, 5], "dropout": 0.3},
    "lr": 1e-3, "batch_size": 64, "max_epochs": 30, "patience": 5,
}

def make_tensors(texts, vocab):
    return torch.tensor([vocab.encode(t, CFG["max_len"]) for t in texts])

@torch.no_grad()
def predict_proba(net, X, bs=512):
    net.eval()
    return torch.cat([torch.sigmoid(net(X[i:i + bs])) for i in range(0, len(X), bs)]).numpy()

def train_cnn(train_texts, y_train, val_texts, y_val, verbose=True):
    set_seed()
    vocab = CharVocab.build(train_texts, CFG["min_char_freq"])
    Xtr, Xva = make_tensors(train_texts, vocab), make_tensors(val_texts, vocab)
    ytr = torch.tensor(y_train.values, dtype=torch.float32)
    net = CharCNN(len(vocab), **CFG["model"])
    pos_weight = (ytr == 0).sum() / (ytr == 1).sum()
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.Adam(net.parameters(), lr=CFG["lr"])
    g = torch.Generator().manual_seed(SEED)
    best_f1, best_state, wait, history = -1, None, 0, []
    for epoch in range(1, CFG["max_epochs"] + 1):
        net.train(); total = 0
        for idx in torch.randperm(len(Xtr), generator=g).split(CFG["batch_size"]):
            opt.zero_grad()
            loss = loss_fn(net(Xtr[idx]), ytr[idx])
            loss.backward(); opt.step(); total += loss.item() * len(idx)
        f1 = prf(y_val, (predict_proba(net, Xva) >= 0.5).astype(int))["f1"]
        history.append({"epoch": epoch, "train_loss": round(total / len(Xtr), 4), "val_f1@0.5": f1})
        if verbose: print(history[-1])
        if f1 > best_f1:
            best_f1, best_state, wait = f1, {k: v.clone() for k, v in net.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= CFG["patience"]:
                if verbose: print("Early stopping"); break
    net.load_state_dict(best_state)
    return net, vocab, history

t0 = time.time()
net, vocab, history = train_cnn(train.norm, train.label, val.norm, val.label)
print(f"Vocabulary size: {len(vocab)} | training time: {time.time() - t0:.0f}s")

## 6. Decision threshold (chosen on validation)

In [ ]:
Xva = make_tensors(val.norm, vocab)
p_val = predict_proba(net, Xva)
grid = np.round(np.arange(0.05, 0.96, 0.05), 2)
scores = pd.DataFrame([{"threshold": t, **prf(val.label, (p_val >= t).astype(int))} for t in grid])
best = scores.loc[scores.f1.idxmax()]
THRESHOLD = float(best.threshold)
print(scores.to_string(index=False))
print("\nChosen threshold:", THRESHOLD)
results["Char-CNN (thr 0.5)"] = prf(val.label, (p_val >= 0.5).astype(int))
results[f"Char-CNN (thr {THRESHOLD})"] = prf(val.label, (p_val >= THRESHOLD).astype(int))

## 7. Extra experiment: effect of normalization
Same model and settings, with lowercased input only (no Kazakh letter mapping, no repeated-letter collapse).

In [ ]:
raw_tr, raw_va = train.text.str.lower(), val.text.str.lower()
net_raw, vocab_raw, _ = train_cnn(raw_tr, train.label, raw_va, val.label, verbose=False)
p_raw = predict_proba(net_raw, make_tensors(raw_va, vocab_raw))
results["Char-CNN, no normalization (thr 0.5)"] = prf(val.label, (p_raw >= 0.5).astype(int))
exp = pd.DataFrame(results).T
exp

## 8. Saving the model, vocabulary, config and validation results

In [ ]:
os.makedirs("model", exist_ok=True); os.makedirs("results", exist_ok=True)
torch.save(net.state_dict(), "model/model.pt")
vocab.save("model/vocab.json")
CFG["threshold"] = THRESHOLD
json.dump(CFG, open("model/config.json", "w"), indent=2)
exp.to_csv("results/validation_experiments.csv")
pd.DataFrame(history).to_csv("results/training_history.csv", index=False)
print(json.dumps(CFG, indent=2))
print("Saved:", os.listdir("model"))